In [8]:
import pandas as pd
import re
import numpy as np

def extract_structured_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extracts high-signal, structured numerical and categorical features 
    from the 'catalog_content' column.

    This function implements Steps 1, 2, and 3 of the feature extraction plan,
    excluding sparse quality flags (Year_Established, Is_Organic, etc.).

    Args:
        df: The input DataFrame (e.g., loaded from train.csv).

    Returns:
        The DataFrame with new structured feature columns added.
    """
    print("Starting Structured Feature Extraction...")

    # --- Helper Functions for Parsing ---

    def safe_extract(pattern, text, default=None):
        """
        Helper to safely extract a regex pattern. 
        It now checks ALL capture groups to correctly handle OR logic (|).
        """
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Iterate through all captured groups and return the first non-None, non-empty value
            for group_value in match.groups():
                if group_value is not None and group_value.strip():
                    return group_value.strip()
        return default

    def parse_value_unit(content):
        """Extracts Value and Unit from the end of catalog_content."""
        # Pattern to find 'Value: X.X\nUnit: Y'
        value_match = re.search(r'Value:\s*([\d\.,]+)', content, re.IGNORECASE)
        unit_match = re.search(r'Unit:\s*([^\n]+)', content, re.IGNORECASE)

        # Handle potential None matches safely
        value = float(value_match.group(1).replace(',', '')) if value_match else np.nan
        unit = unit_match.group(1).strip() if unit_match and unit_match.group(1) is not None else 'Unknown'
        return pd.Series([value, unit])


    # --- Step 1: Core Structured Features (Quantity and Size) ---

    print("Step 1: Extracting Quantity and Size...")

    # Apply the parser to get separate Value and Unit columns
    df[['Total_Declared_Value', 'Unit']] = df['catalog_content'].apply(parse_value_unit)

    # 1.1 Extract Pack_Quantity (e.g., "Pack of 4", "12-Count")
    # Searches the Item Name (first line of catalog_content)
    df['Pack_Quantity'] = df['catalog_content'].apply(lambda x: 
        safe_extract(r'(?:Pack of|Set of|Count of)\s*(\d+)|(\d+)-Pack|(\d+)-Count', x.split('\n', 1)[0])
    )
    # Clean up and convert to integer. If extracted value is None, default to 1.
    df['Pack_Quantity'] = pd.to_numeric(df['Pack_Quantity'], errors='coerce').fillna(1).astype(int)

    # 1.2 Calculate Base_Unit_Value
    # Base_Unit_Value = Total_Declared_Value / Pack_Quantity
    df['Base_Unit_Value'] = df['Total_Declared_Value'] / df['Pack_Quantity']

    # 1.3 Unit Normalization (Example: grouping common units)
    # This is a simplified example; a real-world project would need a full conversion table.
    unit_map = {
        'ounce': 'oz', 'oz.': 'oz', 'fl oz': 'oz', 'fluid ounce': 'oz',
        'pound': 'lb', 'lb.': 'lb',
        'gram': 'g', 'g.': 'g',
        'count': 'count', 'ct': 'count',
        'liters': 'L', 'liter': 'L',
        # Add more units as needed...
    }
    df['Unit_Normalized'] = df['Unit'].str.lower().replace(unit_map, regex=True).fillna('misc')


    # --- Step 2: Brand and Category Features (IMPROVED) ---

    print("Step 2: Extracting Brand and Category (Improved)...")

    # 2.1 Extract Brand Name
    def extract_brand(item_name):
        # Extract the text before the first comma, then take the first word.
        name_parts = item_name.split(',', 1)[0].split()
        return name_parts[0] if name_parts else 'Unknown'
        
    df['Item_Name'] = df['catalog_content'].apply(lambda x: x.split('\n', 1)[0].split(':', 1)[-1].strip())
    df['Brand_Name'] = df['Item_Name'].apply(extract_brand)

    # 2.2 Simple Product Class (IMPROVED - Now uses multiple words)
    def extract_class(item_name):
        parts = item_name.split(',', 1)
        if len(parts) > 1:
            # Take the description AFTER the first comma and clean up parentheses/size info
            description = parts[1].strip()
            
            # Remove content in parentheses (often size info) and "Pack of X"
            description = re.sub(r'\(.*?\)|pack of \d+', '', description, flags=re.IGNORECASE)
            
            # Split and take the first two meaningful words to capture the product type
            words = [word for word in description.split() if len(word) > 1] # Ignore single letters
            
            # Combine up to the first two meaningful words
            if len(words) >= 2:
                return f"{words[0]} {words[1]}"
            if len(words) == 1:
                return words[0]
        return 'Misc'

    df['Product_Class'] = df['Item_Name'].apply(extract_class)


    # --- Step 3: Quality/Context Features (REFINED) ---
    # NOTE: Year_Established, Is_Organic, Is_Original, and Is_Gluten_Free have been removed 
    # as they were highly sparse/low-impact features.

    print("Step 3: Extracting Quality and Contextual Metrics...")

    # 3.1 Bullet Point Count
    # Counting the number of lines starting with 'Bullet Point'
    df['Bullet_Point_Count'] = df['catalog_content'].apply(lambda x: 
        len(re.findall(r'Bullet Point \d+:', x, re.IGNORECASE))
    )

    # 3.2 Description Word Count
    # Total word count of the entire catalog text block
    df['Description_Word_Count'] = df['catalog_content'].apply(lambda x: len(x.split()))

    print("Extraction Complete. New features created.")
    return df

# --- Main Execution Block ---
if __name__ == '__main__':
    # Define the input and output file paths based on your file structure
    INPUT_FILE = '/kaggle/input/csv-file/train.csv'
    OUTPUT_FILE = '/kaggle/working/Updated_train.csv'

    try:
        # Load the raw data
        data_raw = pd.read_csv(INPUT_FILE)
        initial_row_count = len(data_raw)

        # Ensure the 'catalog_content' column is a string for reliable regex
        data_raw['catalog_content'] = data_raw['catalog_content'].astype(str)

        # Extract features
        data_processed = extract_structured_features(data_raw.copy())
        
        final_row_count = len(data_processed)
        
        print(f"\nFinal rows for modeling (All original rows preserved): {final_row_count}")
        
        # Select relevant columns for the modeling team to use
        features_to_save = [
            'sample_id',
            'price', # Keep the target variable
            'image_link', # Keep for Phase 2
            'catalog_content', # Keep for NLP team if needed
            
            # --- STRUCTURED FEATURES ---
            'Total_Declared_Value', 
            'Unit', 
            'Pack_Quantity', 
            'Base_Unit_Value',
            'Unit_Normalized', 
            'Brand_Name', 
            'Product_Class',
            'Bullet_Point_Count', 
            'Description_Word_Count',
        ]

        data_processed[features_to_save].to_csv(OUTPUT_FILE, index=False)
        print(f"\nSuccessfully saved features to {OUTPUT_FILE}")
        print("\nFirst 5 rows of the processed data:")
        print(data_processed[features_to_save].head())

    except FileNotFoundError:
        print(f"Error: Input file '{INPUT_FILE}' not found. Please ensure it is in the correct directory.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


Starting Structured Feature Extraction...
Step 1: Extracting Quantity and Size...
Step 2: Extracting Brand and Category (Improved)...
Step 3: Extracting Quality and Contextual Metrics...
Extraction Complete. New features created.

Final rows for modeling (All original rows preserved): 75000

Successfully saved features to /kaggle/working/Updated_train.csv

First 5 rows of the processed data:
   sample_id  price                                         image_link  \
0      33127   4.89  https://m.media-amazon.com/images/I/51mo8htwTH...   
1     198967  13.12  https://m.media-amazon.com/images/I/71YtriIHAA...   
2     261251   1.97  https://m.media-amazon.com/images/I/51+PFEe-w-...   
3      55858  30.34  https://m.media-amazon.com/images/I/41mu0HAToD...   
4     292686  66.49  https://m.media-amazon.com/images/I/41sA037+Qv...   

                                     catalog_content  Total_Declared_Value  \
0  Item Name: La Victoria Green Taco Sauce Mild, ...                 72.00   
1  I

In [18]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import time
import os
import sys

# --- Configuration ---
INPUT_CSV_PATH = '/kaggle/input/csv-file/train.csv' 
OUTPUT_TENSOR_PATH = 'nlp_embeddings_float16.npy'
# Using the full repository name to aid local cache lookup
MODEL_NAME = 'google-bert/bert-base-uncased' 
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Manual Pooling Function ---
def mean_pooling(model_output, attention_mask):
    """
    Performs Mean Pooling over the token embeddings.
    """
    token_embeddings = model_output[0] 
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) 
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

def generate_and_save_embeddings(df):
    """
    Generates semantic embeddings using BERT, forcing pure OFFLINE mode
    to bypass network blocks. Requires files to be manually in cache.
    """
    
    print(f"Loading model '{MODEL_NAME}' to device: {DEVICE}...")

    # 1. Force OFFLINE mode for the entire execution
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = 'True' 
    
    try:
        # Load the model components using cached files only
        # We explicitly rely on the cache being populated.
        
        # Tokenizer is loaded first and is often the source of the 404 error
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME, 
            trust_remote_code=True, 
            local_files_only=True
        )
        
        # Model loading will fail with 'NoneType' error if pytorch_model.bin is missing
        model = AutoModel.from_pretrained(
            MODEL_NAME, 
            trust_remote_code=True, 
            local_files_only=True
        ).to(DEVICE)
        
        model.eval()
        
    except Exception as e:
        print("\n--- ERROR DURING MODEL LOAD (OFFLINE MODE) ---")
        print(f"Original Error: {e}")
        print("\n*** ACTION REQUIRED ***")
        print("The environment is permanently blocked from downloading model files (Network/Firewall).")
        print("The files for 'bert-base-uncased' are NOT present in your local cache.")
        print("You must manually download the following files and place them in the correct Hugging Face cache folder:")
        print(" - config.json, pytorch_model.bin, tokenizer.json, vocab.txt, tokenizer_config.json")
        print("-------------------------------")
        return

    # --- Processing Logic (Remains unchanged) ---
    sentences = df['catalog_content'].astype(str).tolist() 
    total_samples = len(sentences)
    
    print(f"Found {total_samples} texts to encode. Starting batch processing (Batch Size: {BATCH_SIZE})...")

    all_embeddings = []
    start_time = time.time()
    
    for i in range(0, total_samples, BATCH_SIZE):
        batch = sentences[i:i + BATCH_SIZE]
        
        encoded_input = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(DEVICE) 

        with torch.no_grad():
            model_output = model(**encoded_input)

        sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
        
        all_embeddings.append(sentence_embeddings.cpu().numpy())
        
        if i % (BATCH_SIZE * 10) == 0 and i > 0:
            elapsed = time.time() - start_time
            rate = i / elapsed
            print(f"--- Processed {i}/{total_samples} texts ---")
            print(f"    Time elapsed: {elapsed:.2f}s | Rate: {rate:.2f} samples/s")

    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_embeddings = final_embeddings.astype(np.float16)
    np.save(OUTPUT_TENSOR_PATH, final_embeddings)
    
    final_elapsed = time.time() - start_time
    print("\n--- Encoding Complete ---")
    print(f"Final tensor shape: {final_embeddings.shape}")
    print(f"Total time taken: {final_elapsed:.2f} seconds.")
    print(f"The semantic features (F_text^semantic) are saved to '{OUTPUT_TENSOR_PATH}'.")


if __name__ == '__main__':
    if not os.path.exists(INPUT_CSV_PATH):
        print(f"Error: Input CSV not found at '{INPUT_CSV_PATH}'.")
        print("Please ensure 'feature_extractor.py' has been run successfully.")
    else:
        # Load the CSV containing the catalog content
        df = pd.read_csv(INPUT_CSV_PATH)
        generate_and_save_embeddings(df)


Loading model 'google-bert/bert-base-uncased' to device: cuda...

--- ERROR DURING MODEL LOAD (OFFLINE MODE) ---
Original Error: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

*** ACTION REQUIRED ***
The environment is permanently blocked from downloading model files (Network/Firewall).
The files for 'bert-base-uncased' are NOT present in your local cache.
You must manually download the following files and place them in the correct Hugging Face cache folder:
 - config.json, pytorch_model.bin, tokenizer.json, vocab.txt, tokenizer_config.json
-------------------------------


 #                             Harshith's code modification 

In [48]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18 
from sentence_transformers import SentenceTransformer
import pandas as pd
from tqdm import tqdm
import requests
from io import BytesIO
import os
from sklearn.model_selection import train_test_split
import numpy as np

# --- Configuration Constants ---
TEXT_MODEL_NAME = 'all-MiniLM-L6-v2'
TEXT_EMBEDDING_PATH = 'precomputed_text_embeddings.pt'
TRAIN_CSV_PATH = '/kaggle/input/csv-file/train.csv'
IMAGE_FOLDER_PATH = '/kaggle/input/amzn-ml-challenge-images/images'
BATCH_SIZE = 16
EPOCHS = 5

# ==============================================================================
# 0. STANDALONE ENCODING FUNCTION (To be run ONCE)
# ==============================================================================

def create_and_save_embeddings(csv_file, model_name, output_path):
    """
    Loads data, encodes text features using SentenceTransformer, and saves them to disk.
    This function is isolated to handle the network-sensitive model loading only once.
    """
    if os.path.exists(output_path):
        print(f"Text embeddings found at '{output_path}'. Skipping creation.")
        return

    print(f"--- Creating and Saving Text Embeddings for {model_name} ---")
    df = pd.read_csv(csv_file)
    sentences = df['catalog_content'].astype(str).tolist()
    
    # --- Aggressive Offline/Fallback Model Load ---
    try:
        # Attempt to load, allowing network fallback if local fails
        text_model = SentenceTransformer(model_name)
    except Exception as e:
        print("\n*** CRITICAL FAILURE: MODEL DOWNLOAD BLOCKED ***")
        print(f"Model loading failed due to network restriction/cache miss: {e}")
        print("Please ensure the model files are manually placed in the Hugging Face cache.")
        return # Exit the function if model cannot be loaded

    # --- Encoding ---
    print("Encoding text embeddings (this may take a few minutes)...")
    # Batch encoding is significantly faster than the iterative loop used before
    embeddings = text_model.encode(sentences, show_progress_bar=True, convert_to_tensor=True, device='cuda' if torch.cuda.is_available() else 'cpu')
    
    # Save the tensor
    torch.save(embeddings.cpu(), output_path)
    print(f"\nSuccessfully saved {embeddings.shape[0]} embeddings to '{output_path}'.")


# ==============================================================================
# 1. DATASET
# ==============================================================================

class PriceDataset(Dataset):
    # ACCEPT the path to the pre-computed embeddings file
    def __init__(self, csv_file, image_folder=None, mode='train', embedding_path=TEXT_EMBEDDING_PATH):
        
        self.mode = mode
        self.df = pd.read_csv(csv_file)
        self.image_folder = image_folder

        # --- Text embeddings ---
        # IMPROVEMENT: Load embeddings directly from disk, bypassing network/encoding during training
        try:
            self.text_embeddings = torch.load(embedding_path)
            print(f"Loaded text embeddings from {embedding_path}. Shape: {self.text_embeddings.shape}")
            # Get input dimension for the TextEncoder from the loaded data
            self.text_input_dim = self.text_embeddings.shape[1]
        except Exception as e:
            print(f"FATAL: Could not load embeddings from {embedding_path}. Run create_and_save_embeddings first.")
            raise e
        
        # --- Image transformations ---
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

        # --- Labels ---
        if mode == 'train':
            # Labels are stored as a tensor
            self.labels = torch.tensor(self.df['price'].values, dtype=torch.float32)
        else:
            self.labels = None

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- Load Image (Only from disk or placeholder) ---
        if self.image_folder and os.path.exists(self.image_folder):
            img_name = f"{row['sample_id']}.jpg" 
            img_path = os.path.join(self.image_folder, img_name)
            if os.path.exists(img_path):
                img = Image.open(img_path).convert('RGB')
            else:
                img = Image.new('RGB', (224,224))  
        else:
            img = Image.new('RGB', (224,224)) 

        img = self.transform(img)

        # --- Features ---
        text_emb = self.text_embeddings[idx]

        sample_id = row['sample_id']

        if self.mode == 'train':
            return img, text_emb, self.labels[idx], sample_id
        else:
            return img, text_emb, sample_id



In [42]:
# ==============================================================================
# 2. MODEL ARCHITECTURE (Encoders, Fusion, Predictor)
# ==============================================================================

class ImageEncoder(nn.Module):
    def __init__(self, output_dim=512):
        super().__init__()
        self.backbone = resnet18(pretrained=True)
        self.backbone.fc = nn.Identity() 
        self.fc = nn.Linear(512, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        x = self.norm(x)
        return x.unsqueeze(1) 

class TextEncoder(nn.Module):
    # input_dim must be dynamic, based on the loaded embedding size
    def __init__(self, input_dim, output_dim=512):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        self.relu = nn.ReLU() 
        self.norm = nn.LayerNorm(output_dim)
        
    def forward(self, x):
        x = self.fc(x)
        x = self.relu(x)
        x = self.norm(x)
        return x.unsqueeze(1)

class CrossAttentionFusion(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        self.fusion_fc = nn.Linear(d_model * 2, d_model)
        self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, img_feat, text_feat):
        img_feat = img_feat.squeeze(1)
        text_feat = text_feat.squeeze(1)
        combined = torch.cat([img_feat, text_feat], dim=1) 
        fused = self.fusion_fc(combined)
        fused = self.relu(fused)
        fused = self.norm(fused)
        
        return fused 

class PricePredictor(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Linear(d_model//2, 1),
            nn.Softplus()
        )
        
    def forward(self, x):
        return self.mlp(x)

class MultiModalPriceModel(nn.Module):
    # ACCEPT text_input_dim dynamically from the Dataset
    def __init__(self, text_input_dim):
        super().__init__()
        self.img_encoder = ImageEncoder()
        # Pass the dynamic input dimension
        self.text_encoder = TextEncoder(input_dim=text_input_dim) 
        self.fusion = CrossAttentionFusion() 
        self.predictor = PricePredictor()
        
    def forward(self, img, text_emb):
        img_feat = self.img_encoder(img) 
        text_feat = self.text_encoder(text_emb)
        fused = self.fusion(img_feat, text_feat) 
        price = self.predictor(fused) 
        return price


In [43]:
# ==============================================================================
# 3. LOSS FUNCTION
# ==============================================================================

def smape_loss(y_pred, y_true, eps=1e-6):
    """
    Symmetric Mean Absolute Percentage Error (SMAPE) loss function.
    """
    return torch.mean(2 * torch.abs(y_pred - y_true) / (torch.abs(y_pred) + torch.abs(y_true) + eps))

In [46]:


# ==============================================================================
# 4. TRAINING AND EVALUATION (Example Usage)
# ==============================================================================

# --- Setup Data Loaders ---
# NOTE: Replace with your actual file paths
try:
    full_dataset = PriceDataset(
        '/kaggle/input/csv-file/train.csv',  
        image_folder='dataset/images',  
        mode='train'
    )

    train_indices, val_indices = train_test_split(
        list(range(len(full_dataset))),
        test_size=0.2, 
        random_state=42
    )

    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- Training Loop ---
    epochs = 5
    model = MultiModalPriceModel().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1} Training")
        
        for imgs, text_emb, labels, _ in train_loop:
            imgs, text_emb, labels = imgs.to(device), text_emb.to(device), labels.to(device).unsqueeze(1) # unsqueeze label
            optimizer.zero_grad()
            preds = model(imgs, text_emb)
            loss = smape_loss(preds, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_loop.set_postfix(train_loss=train_loss/(train_loop.n+1))
        
        # Validation
        model.eval()
        val_loss = 0
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1} Validation")
        
        with torch.no_grad():
            for imgs, text_emb, labels, _ in val_loop:
                imgs, text_emb, labels = imgs.to(device), text_emb.to(device), labels.to(device).unsqueeze(1) # unsqueeze label
                preds = model(imgs, text_emb)
                loss = smape_loss(preds, labels)
                val_loss += loss.item()
                val_loop.set_postfix(val_loss=val_loss/(val_loop.n+1))
        
        print(f"Epoch {epoch+1} -> Avg Train Loss: {train_loss/len(train_loader):.4f}, Avg Val Loss: {val_loss/len(val_loader):.4f}")

    # --- Prediction Setup ---
    # NOTE: The subsequent prediction and scoring logic remains largely the same,
    # but the label must be explicitly unsqueezed during training/validation.

    # Placeholder for prediction and scoring logic to show completeness
    def smape(y_true, y_pred, eps=1e-6):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + eps))
        
    print("\nTraining complete. Prediction and scoring logic follows...")

except FileNotFoundError as e:
    print(f"\nFATAL ERROR: A required file was not found: {e.filename}")
    print("Please verify the paths for 'sample_train.csv' and 'dataset/images'.")
except Exception as e:
    print(f"\nAn error occurred during execution: {e}")
    print("Please check model loading and data paths.")


An error occurred during execution: 404 Client Error. (Request ID: Root=1-68ea60b2-23d57a0a227784e57c6ae321;a6015c5e-0202-4e37-bf90-7979b79ab67b)

Entry Not Found for url: https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false.
additional_chat_templates does not exist on "main"
Please check model loading and data paths.
